In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
from faker import Faker
fake = Faker()
print(fake.name())

In [0]:
import uuid
import random
from faker import Faker

fake = Faker()

ACCOUNT_TYPES = ["checking", "savings", "loan"]

def make_account_id() -> str:
    return f"ACC-{uuid.uuid4().hex[:12].upper()}"

def make_record():
    return {
        "account_id": make_account_id(),
        "customer_ssn": fake.ssn(),
        "customer_name": fake.name(),
        "balance": round(random.uniform(0, 250_000), 2),
        "account_type": random.choice(ACCOUNT_TYPES),
    }

def inject_errors(records):
    if len(records) < 3:
        return records
    records[0]["customer_ssn"] = None  # null SSN
    source_ssn = records[2]["customer_ssn"]
    records[1]["customer_ssn"] = source_ssn  # duplicate SSN
    return records

In [0]:
ERROR_RATE = 0.0002  # ~0.02% chance per record - most batches clean, occasional real failure (~14% of batches)

COUNT = random.randint(500, 1000)
records = [make_record() for _ in range(COUNT)]

error_count = 0
for i, record in enumerate(records):
    if random.random() < ERROR_RATE:
        issue_type = random.choice(["null_ssn", "duplicate_ssn"])
        if issue_type == "null_ssn":
            record["customer_ssn"] = None
        elif issue_type == "duplicate_ssn" and i > 0:
            record["customer_ssn"] = records[i - 1]["customer_ssn"]
        error_count += 1

print(f"Generated {len(records)} records")
print(f"Naturally occurring data quality issues this run: {error_count}")

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

batch_id = datetime.now(timezone.utc).isoformat()

spark.sql("CREATE CATALOG IF NOT EXISTS banking_contracts_demo")
spark.sql("CREATE SCHEMA IF NOT EXISTS banking_contracts_demo.bronze")

df = spark.createDataFrame(records).withColumn("batch_id", F.lit(batch_id))

df.write.mode("append").saveAsTable("banking_contracts_demo.bronze.accounts_daily")

print(f"Appended batch {batch_id} to banking_contracts_demo.bronze.accounts_daily")
display(df.limit(10))